In [3]:
# -*- coding: utf-8 -*-
"""
2D staged fully learned DAE (FL-DAE), corrected to match the derived asymptotic system.

Asymptotic system used in this script
--------------------------------------
Outer states:
    phi^pm * (phi^pm_x + phi^pm_y) = f(x,y).
Interface:
    h_t = 0.5 * (h_y - 1) * (phi^-(h,y) + phi^+(h,y)).
Inner corrections:
    Q^pm_xixi
    + [h_t + (phi^pm(h,y) + Q^pm)(1-h_y)]/sqrt(1+h_y^2) * Q^pm_xi = 0.
Matching at xi=0:
    phi^-(h,y)+Q^-(0) = phi^+(h,y)+Q^+(0)
                       = 0.5*(phi^-(h,y)+phi^+(h,y)).
Far field (truncated):
    Q^-(-XI_MAX)=0 and Q^+(XI_MAX)=0, imposed as hard constraints.
Reconstruction coordinate:
    xi = (x-h(y,t))*sqrt(1+h_y(y,t)^2)/mu.

No independent periodic loss is imposed on Q.  The only soft periodic losses
are those explicitly present in the derived outer and interface problems.
"""

from __future__ import annotations

import math
import os
import random
import time
from typing import Dict, List, Sequence, Tuple, Union

import numpy as np
import pandas as pd                                     
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc

Tensor = torch.Tensor
TensorInputs = Union[Tensor, Sequence[Tensor]]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0
L_VALUE, R_VALUE = -4.0, 2.0
H0_VALUE = 0.0

DEPTH, WIDTH = 5, 20
LR = 1.0e-3
XI_MAX = 12.0
W_PHI_PER = 1.0
W_H_PER = 1.0

NUM_SAMPLES = 10000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
BASE_PATH = "."
SAVE_LHS_PREDICTION = True
CHECK_EVERY = 100

PI = torch.tensor(math.pi, dtype=DTYPE, device=DEVICE)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def mse(x: Tensor) -> Tensor:
    return torch.mean(x.square())


def mean_std(values: Sequence[float]) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))


def all_grads(
    outputs: Tensor,
    inputs: TensorInputs,
    create_graph: bool = True,
    retain_graph: bool | None = None,
):
    """Return every requested gradient; never discard tuple components."""
    if retain_graph is None:
        retain_graph = create_graph
    single = isinstance(inputs, torch.Tensor)
    input_tuple = (inputs,) if single else tuple(inputs)
    values = torch.autograd.grad(
        outputs=outputs,
        inputs=input_tuple,
        grad_outputs=torch.ones_like(outputs),
        create_graph=create_graph,
        retain_graph=retain_graph,
        only_inputs=True,
        allow_unused=False,
    )
    return values[0] if single else values


def source_f(x: Tensor, y: Tensor) -> Tensor:
    return torch.cos(PI * x / 4.0) * torch.cos(PI * y / 4.0)


class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, width: int, depth: int):
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                init.xavier_normal_(layer.weight)
                if layer.bias is not None:
                    init.zeros_(layer.bias)

    def forward(self, z: Tensor) -> Tensor:
        return self.net(z)


class ThreeModuleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.N_phi = MLP(2, 2, WIDTH, DEPTH)
        self.N_h = MLP(2, 1, WIDTH, DEPTH)
        self.N_Q = MLP(4, 2, WIDTH, DEPTH)

    def phi_m(self, x: Tensor, y: Tensor) -> Tensor:
        raw = self.N_phi(torch.cat([x, y], dim=1))[:, 0:1]
        return L_VALUE + (x - X_MIN) * raw

    def phi_p(self, x: Tensor, y: Tensor) -> Tensor:
        raw = self.N_phi(torch.cat([x, y], dim=1))[:, 1:2]
        return R_VALUE + (x - X_MAX) * raw

    def h(self, y: Tensor, t: Tensor) -> Tensor:
        return H0_VALUE + t * self.N_h(torch.cat([y, t], dim=1))

    def Q_m(self, xi: Tensor, h: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, y, t], dim=1))[:, 0:1]
        return ((xi + XI_MAX) / XI_MAX) * raw

    def Q_p(self, xi: Tensor, h: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, y, t], dim=1))[:, 1:2]
        return ((XI_MAX - xi) / XI_MAX) * raw


def set_trainable(model: ThreeModuleModel, phi: bool, h: bool, Q: bool) -> None:
    for p in model.N_phi.parameters():
        p.requires_grad_(phi)
    for p in model.N_h.parameters():
        p.requires_grad_(h)
    for p in model.N_Q.parameters():
        p.requires_grad_(Q)


def sample_x(n: int) -> Tensor:
    return X_MIN + (X_MAX - X_MIN) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)


def sample_y(n: int) -> Tensor:
    return Y_MIN + (Y_MAX - Y_MIN) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)


def sample_t(n: int) -> Tensor:
    return T_FINAL * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)


def sample_xi_m(n: int) -> Tensor:
    return -XI_MAX * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)


def sample_xi_p(n: int) -> Tensor:
    return XI_MAX * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)


def constant_y(value: float, n: int) -> Tensor:
    return torch.full((n, 1), value, device=DEVICE, dtype=DTYPE)


# -----------------------------------------------------------------------------
# Derived losses
# -----------------------------------------------------------------------------
def outer_loss(model: ThreeModuleModel, x: Tensor, y: Tensor, x_per: Tensor):
    pm = model.phi_m(x, y)
    pp = model.phi_p(x, y)
    pm_x, pm_y = all_grads(pm, (x, y), create_graph=True, retain_graph=True)
    pp_x, pp_y = all_grads(pp, (x, y), create_graph=True, retain_graph=True)
    f = source_f(x, y)
    loss_res = mse(pm * (pm_x + pm_y) - f) + mse(pp * (pp_x + pp_y) - f)

    y_l = constant_y(Y_MIN, x_per.shape[0])
    y_r = constant_y(Y_MAX, x_per.shape[0])
    loss_per = (
        mse(model.phi_m(x_per, y_l) - model.phi_m(x_per, y_r))
        + mse(model.phi_p(x_per, y_l) - model.phi_p(x_per, y_r))
    )
    return loss_res + W_PHI_PER * loss_per, loss_res, loss_per


def h_loss(model: ThreeModuleModel, y: Tensor, t: Tensor, t_per: Tensor):
    h_val = model.h(y, t)
    h_t, h_y = all_grads(h_val, (t, y), create_graph=True, retain_graph=True)
    pm_h = model.phi_m(h_val, y)
    pp_h = model.phi_p(h_val, y)
    residual = h_t - 0.5 * (h_y - 1.0) * (pm_h + pp_h)
    loss_res = mse(residual)

    y_l = constant_y(Y_MIN, t_per.shape[0])
    y_r = constant_y(Y_MAX, t_per.shape[0])
    loss_per = mse(model.h(y_l, t_per) - model.h(y_r, t_per))
    return loss_res + W_H_PER * loss_per, loss_res, loss_per


def frozen_h_data(model: ThreeModuleModel, y_base: Tensor, t_base: Tensor):
    y = y_base.detach().clone().requires_grad_(True)
    t = t_base.detach().clone().requires_grad_(True)
    h_val = model.h(y, t)
    h_y, h_t = all_grads(
        h_val, (y, t), create_graph=False, retain_graph=False
    )
    return h_val.detach(), h_y.detach(), h_t.detach()


def q_residual_frozen(
    model: ThreeModuleModel,
    side: str,
    xi_base: Tensor,
    y_base: Tensor,
    t_base: Tensor,
    training_graph: bool,
) -> Tensor:
    """Q residual for staged FL-DAE or for RAR candidate evaluation."""
    h, h_y, h_t = frozen_h_data(model, y_base, t_base)
    y = y_base.detach()
    t = t_base.detach()
    phi = (model.phi_m(h, y) if side == "m" else model.phi_p(h, y)).detach()
    metric = torch.sqrt(1.0 + h_y.square())

    q_input = torch.cat([xi_base.detach(), h, y, t], dim=1).detach().requires_grad_(True)
    xi = q_input[:, 0:1]
    h_q = q_input[:, 1:2]
    y_q = q_input[:, 2:3]
    t_q = q_input[:, 3:4]
    Q = model.Q_m(xi, h_q, y_q, t_q) if side == "m" else model.Q_p(xi, h_q, y_q, t_q)

    # First derivative must create a graph even during candidate evaluation,
    # otherwise Q_xixi cannot be formed.
    q_grad = all_grads(Q, q_input, create_graph=True, retain_graph=True)
    Q_xi = q_grad[:, 0:1]
    q2_grad = all_grads(
        Q_xi,
        q_input,
        create_graph=training_graph,
        retain_graph=training_graph,
    )
    Q_xixi = q2_grad[:, 0:1]
    coeff = (h_t + (phi + Q) * (1.0 - h_y)) / metric
    residual = Q_xixi + coeff * Q_xi
    return residual if training_graph else residual.detach()


def q_match_frozen(model: ThreeModuleModel, y_base: Tensor, t_base: Tensor) -> Tensor:
    h, _, _ = frozen_h_data(model, y_base, t_base)
    y = y_base.detach()
    t = t_base.detach()
    pm = model.phi_m(h, y).detach()
    pp = model.phi_p(h, y).detach()
    middle = 0.5 * (pm + pp)
    xi0 = torch.zeros_like(t)
    qm = model.Q_m(xi0, h, y, t)
    qp = model.Q_p(xi0, h, y, t)
    return mse(pm + qm - middle) + mse(pp + qp - middle)


def q_loss_frozen(
    model: ThreeModuleModel,
    xi_m: Tensor, y_m: Tensor, t_m: Tensor,
    xi_p: Tensor, y_p: Tensor, t_p: Tensor,
    y_match: Tensor, t_match: Tensor,
):
    rm = q_residual_frozen(model, "m", xi_m, y_m, t_m, training_graph=True)
    rp = q_residual_frozen(model, "p", xi_p, y_p, t_p, training_graph=True)
    loss_res = mse(rm) + mse(rp)
    loss_match = q_match_frozen(model, y_match, t_match)
    return loss_res + loss_match, loss_res, loss_match


def q_residual_joint(
    model: ThreeModuleModel,
    side: str,
    xi: Tensor,
    y: Tensor,
    t: Tensor,
) -> Tensor:
    """End-to-end Q residual for BL-PINN joint training."""
    h = model.h(y, t)
    h_y, h_t = all_grads(h, (y, t), create_graph=True, retain_graph=True)
    phi = model.phi_m(h, y) if side == "m" else model.phi_p(h, y)
    metric = torch.sqrt(1.0 + h_y.square())
    Q = model.Q_m(xi, h, y, t) if side == "m" else model.Q_p(xi, h, y, t)
    Q_xi = all_grads(Q, xi, create_graph=True, retain_graph=True)
    Q_xixi = all_grads(Q_xi, xi, create_graph=True, retain_graph=True)
    coeff = (h_t + (phi + Q) * (1.0 - h_y)) / metric
    return Q_xixi + coeff * Q_xi


def q_match_joint(model: ThreeModuleModel, y: Tensor, t: Tensor) -> Tensor:
    h = model.h(y, t)
    pm = model.phi_m(h, y)
    pp = model.phi_p(h, y)
    middle = 0.5 * (pm + pp)
    xi0 = torch.zeros_like(t)
    qm = model.Q_m(xi0, h, y, t)
    qp = model.Q_p(xi0, h, y, t)
    return mse(pm + qm - middle) + mse(pp + qp - middle)


def q_loss_joint(
    model: ThreeModuleModel,
    xi_m: Tensor, y_m: Tensor, t_m: Tensor,
    xi_p: Tensor, y_p: Tensor, t_p: Tensor,
    y_match: Tensor, t_match: Tensor,
):
    rm = q_residual_joint(model, "m", xi_m, y_m, t_m)
    rp = q_residual_joint(model, "p", xi_p, y_p, t_p)
    loss_res = mse(rm) + mse(rp)
    loss_match = q_match_joint(model, y_match, t_match)
    return loss_res + loss_match, loss_res, loss_match


# -----------------------------------------------------------------------------
# Reconstruction
# -----------------------------------------------------------------------------
def reconstruct(model: ThreeModuleModel, x: Tensor, y: Tensor, t: Tensor, mu: float) -> Tensor:
    with torch.enable_grad():
        y_in = y.detach().clone().requires_grad_(True)
        t_in = t.detach()
        h_val = model.h(y_in, t_in)
        h_y = all_grads(h_val, y_in, create_graph=False, retain_graph=False)

    with torch.no_grad():
        h = h_val.detach()
        metric = torch.sqrt(1.0 + h_y.detach().square())
        # Correct normal stretched coordinate from the asymptotic derivation.
        xi = (x - h) * metric / float(mu)
        xi_m = torch.clamp(xi, min=-XI_MAX, max=0.0)
        xi_p = torch.clamp(xi, min=0.0, max=XI_MAX)
        pm = model.phi_m(x, y)
        pp = model.phi_p(x, y)
        qm = model.Q_m(xi_m, h, y, t)
        qp = model.Q_p(xi_p, h, y, t)
        return torch.where(x <= h, pm + qm, pp + qp)


# -----------------------------------------------------------------------------
# Reference data and common LHS testing set
# -----------------------------------------------------------------------------
def get_target_col(df: pd.DataFrame) -> str:
    if "u" in df.columns:
        return "u"
    if "u0" in df.columns:
        return "u0"
    return str(df.columns[-1])


def load_reference(mu: float):
    mu_id = round(-math.log10(mu))
    filename = f"2d_U0_all_t_u_x_y_t_mu{mu_id}_101_101_101_Mathematica_620.csv"
    path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find reference file: {filename}")
    df = pd.read_csv(path)
    df.columns = [str(c).lower().strip() for c in df.columns]
    df = df.sort_values(["t", "x", "y"]).reset_index(drop=True)
    return df, filename


def generate_lhs_indices(df: pd.DataFrame, mu: float) -> np.ndarray:
    if len(df) < NUM_SAMPLES:
        raise ValueError("Reference grid has fewer points than NUM_SAMPLES.")
    mins = [df["t"].min(), df["x"].min(), df["y"].min()]
    maxs = [df["t"].max(), df["x"].max(), df["y"].max()]
    tree = cKDTree(df[["t", "x", "y"]].values)
    selected: List[int] = []
    used = set()
    for batch_id in range(100):
        sampler = qmc.LatinHypercube(d=3, seed=LHS_SEED + batch_id)
        points = qmc.scale(sampler.random(NUM_SAMPLES), mins, maxs)
        _, indices = tree.query(points)
        for idx in indices:
            idx = int(idx)
            if idx not in used:
                used.add(idx)
                selected.append(idx)
                if len(selected) == NUM_SAMPLES:
                    break
        if len(selected) == NUM_SAMPLES:
            break
    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(np.arange(len(df)), np.asarray(selected, dtype=int))
        fill = np.random.default_rng(LHS_SEED).choice(
            remaining, NUM_SAMPLES - len(selected), replace=False
        )
        selected.extend(int(v) for v in fill)
    result = np.asarray(selected, dtype=int)
    if len(result) != NUM_SAMPLES or len(np.unique(result)) != NUM_SAMPLES:
        raise RuntimeError("Failed to construct exactly NUM_SAMPLES unique test points.")
    np.save(f"2d_LHS_sample_indices_mu{mu:.0e}.npy", result)
    return result


def build_lhs(mu: float) -> Dict[str, object]:
    df, filename = load_reference(mu)
    index_file = f"2d_LHS_sample_indices_mu{mu:.0e}.npy"
    indices = None
    if os.path.exists(index_file):
        loaded = np.load(index_file)
        valid = (
            len(loaded) == NUM_SAMPLES
            and len(np.unique(loaded)) == NUM_SAMPLES
            and np.min(loaded) >= 0
            and np.max(loaded) < len(df)
        )
        if valid:
            indices = loaded
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")
    if indices is None:
        indices = generate_lhs_indices(df, mu)
        print(f"[mu={mu}] Generated {len(indices)} LHS indices from {filename}.")
    t_np = df.iloc[indices]["t"].to_numpy().reshape(-1, 1)
    x_np = df.iloc[indices]["x"].to_numpy().reshape(-1, 1)
    y_np = df.iloc[indices]["y"].to_numpy().reshape(-1, 1)
    u_np = df.iloc[indices][get_target_col(df)].to_numpy().reshape(-1)
    return {
        "t": torch.tensor(t_np, dtype=DTYPE, device=DEVICE),
        "x": torch.tensor(x_np, dtype=DTYPE, device=DEVICE),
        "y": torch.tensor(y_np, dtype=DTYPE, device=DEVICE),
        "true": u_np,
        "t_np": t_np, "x_np": x_np, "y_np": y_np,
        "n_test": len(indices),
    }


def errors(true: np.ndarray, pred: np.ndarray) -> Tuple[float, float]:
    diff = pred - true
    denominator = np.linalg.norm(true)
    if denominator == 0:
        raise ZeroDivisionError("Reference vector has zero norm.")
    return float(np.linalg.norm(diff) / denominator), float(np.max(np.abs(diff)))


def timed_evaluation(model: ThreeModuleModel, data: Dict[str, object], mu: float):
    x, y, t = data["x"], data["y"], data["t"]
    for _ in range(EVAL_WARMUP):
        _ = reconstruct(model, x, y, t, mu)
    sync_cuda()
    start = time.perf_counter()
    for _ in range(EVAL_REPEAT):
        _ = reconstruct(model, x, y, t, mu)
    sync_cuda()
    t_eval = (time.perf_counter() - start) / EVAL_REPEAT
    pred = reconstruct(model, x, y, t, mu).detach().cpu().numpy().reshape(-1)
    e2, einf = errors(data["true"], pred)
    return t_eval, pred, e2, einf


def save_prediction(prefix: str, data: Dict[str, object], pred: np.ndarray) -> None:
    if not SAVE_LHS_PREDICTION:
        return
    pd.DataFrame({
        "t": data["t_np"].reshape(-1),
        "x": data["x_np"].reshape(-1),
        "y": data["y_np"].reshape(-1),
        "u": pred,
    }).to_csv(prefix, index=False)


# =============================================================================
# Staged FL-DAE without RAR
# =============================================================================
METHOD_NAME = "FLDAE"
ITERS_PHI, ITERS_H, ITERS_Q = 5000, 10000, 15000
N_PHI, N_PHI_P = 3000, 1500
N_H, N_H_P = 3000, 1500
N_Q, N_M = 4000, 2000
TOTAL_STAGE_ITERS = ITERS_PHI + ITERS_H + ITERS_Q
TOTAL_LOSS_POINT_STEPS = (
    ITERS_PHI * (N_PHI + 2 * N_PHI_P)
    + ITERS_H * (N_H + 2 * N_H_P)
    + ITERS_Q * (2 * N_Q + N_M)
)


def train_phi(model, x_base, y_base, x_per):
    set_trainable(model, True, False, False)
    optimizer = optim.Adam(model.N_phi.parameters(), lr=LR)
    history = []
    sync_cuda(); start = time.perf_counter()
    for _ in range(ITERS_PHI):
        optimizer.zero_grad(set_to_none=True)
        x = x_base.detach().clone().requires_grad_(True)
        y = y_base.detach().clone().requires_grad_(True)
        loss, _, _ = outer_loss(model, x, y, x_per)
        loss.backward(); optimizer.step(); history.append(loss.detach())
    sync_cuda()
    return time.perf_counter() - start, torch.stack(history)


def train_h(model, y_base, t_base, t_per):
    set_trainable(model, False, True, False)
    optimizer = optim.Adam(model.N_h.parameters(), lr=LR)
    history = []
    sync_cuda(); start = time.perf_counter()
    for _ in range(ITERS_H):
        optimizer.zero_grad(set_to_none=True)
        y = y_base.detach().clone().requires_grad_(True)
        t = t_base.detach().clone().requires_grad_(True)
        loss, _, _ = h_loss(model, y, t, t_per)
        loss.backward(); optimizer.step(); history.append(loss.detach())
    sync_cuda()
    return time.perf_counter() - start, torch.stack(history)


def train_Q(model, xi_m, y_m, t_m, xi_p, y_p, t_p, y_match, t_match):
    set_trainable(model, False, False, True)
    optimizer = optim.Adam(model.N_Q.parameters(), lr=LR)
    history = []
    sync_cuda(); start = time.perf_counter()
    for _ in range(ITERS_Q):
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = q_loss_frozen(
            model, xi_m, y_m, t_m, xi_p, y_p, t_p, y_match, t_match
        )
        loss.backward(); optimizer.step(); history.append(loss.detach())
    sync_cuda()
    return time.perf_counter() - start, torch.stack(history)


def main():
    print("=" * 88)
    print("2D FL-DAE: derived three-stage system, no independent Q-periodic loss")
    print(f"Device={DEVICE}, N_test={NUM_SAMPLES}")
    print("=" * 88)
    lhs = {mu: build_lhs(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for seed in SEEDS:
        set_seed(seed)
        x_phi, y_phi, x_phi_p = sample_x(N_PHI), sample_y(N_PHI), sample_x(N_PHI_P)
        y_h, t_h, t_h_p = sample_y(N_H), sample_t(N_H), sample_t(N_H_P)
        xi_m, y_m, t_m = sample_xi_m(N_Q), sample_y(N_Q), sample_t(N_Q)
        xi_p, y_p, t_p = sample_xi_p(N_Q), sample_y(N_Q), sample_t(N_Q)
        y_match, t_match = sample_y(N_M), sample_t(N_M)

        model = ThreeModuleModel().to(DEVICE)
        T_phi, hist_phi = train_phi(model, x_phi, y_phi, x_phi_p)
        T_h, hist_h = train_h(model, y_h, t_h, t_h_p)
        T_Q, hist_Q = train_Q(
            model, xi_m, y_m, t_m, xi_p, y_p, t_p, y_match, t_match
        )
        T_train = T_phi + T_h + T_Q
        hp = hist_phi.cpu().numpy().astype(np.float64)
        hh = hist_h.cpu().numpy().astype(np.float64)
        hq = hist_Q.cpu().numpy().astype(np.float64)
        np.save(f"2d_FLDAE_loss_phi_seed{seed}.npy", hp)
        np.save(f"2d_FLDAE_loss_h_seed{seed}.npy", hh)
        np.save(f"2d_FLDAE_loss_Q_seed{seed}.npy", hq)
        e_loss = float(hp[-1] + hh[-1] + hq[-1])
        print(f"seed={seed}: T_phi={T_phi:.2f}, T_h={T_h:.2f}, T_Q={T_Q:.2f}, loss={e_loss:.3e}")

        model.eval(); set_trainable(model, False, False, False)
        for mu in MU_LIST:
            T_eval, pred, e2, einf = timed_evaluation(model, lhs[mu], mu)
            metrics[mu].append({
                "Seed": seed, "N_test": lhs[mu]["n_test"],
                "loss_phi": float(hp[-1]), "loss_h": float(hh[-1]),
                "loss_Q": float(hq[-1]), "e_loss": e_loss,
                "e2": e2, "einf": einf,
                "T_phi": T_phi, "T_h": T_h, "T_Q": T_Q,
                "T_train": T_train, "T_eval": T_eval, "T_total": T_train + T_eval,
                "T_train_per_stage_iter_ms": 1e3 * T_train / TOTAL_STAGE_ITERS,
                "T_train_per_loss_point_us": 1e6 * T_train / TOTAL_LOSS_POINT_STEPS,
                "total_stage_iters": TOTAL_STAGE_ITERS,
                "total_loss_point_steps": TOTAL_LOSS_POINT_STEPS,
                "final_Q_loss_points": 2 * N_Q + N_M,
            })
            print(f"  mu={mu}: T_eval={T_eval:.6e}, e2={e2:.3e}, einf={einf:.3e}")
            save_prediction(
                f"2d_FLDAE_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv", lhs[mu], pred
            )

    for mu in MU_LIST:
        df = pd.DataFrame(metrics[mu])
        df.to_csv(f"2d_FLDAE_mu{mu:.0e}_Metrics_Summary.csv", index=False)
        print(f"\n### 2D FL-DAE, mu={mu} [mean +/- sample std] ###")
        for col in ["loss_phi", "loss_h", "loss_Q", "e2", "einf", "T_train", "T_eval", "T_total"]:
            m, s = mean_std(df[col]); print(f"{col}: {m:.6e} +/- {s:.6e}")


if __name__ == "__main__":
    main()


2D FL-DAE: derived three-stage system, no independent Q-periodic loss
Device=cuda, N_test=10000
[mu=0.01] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-02.npy.
[mu=0.001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-03.npy.
[mu=0.0001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-04.npy.
seed=33: T_phi=103.89, T_h=135.54, T_Q=620.38, loss=3.966e-03
  mu=0.01: T_eval=2.783182e-03, e2=5.085e-02, einf=5.767e+00
  mu=0.001: T_eval=2.403374e-03, e2=7.162e-02, einf=6.326e+00
  mu=0.0001: T_eval=2.443994e-03, e2=7.354e-02, einf=6.395e+00
seed=99: T_phi=104.26, T_h=135.08, T_Q=615.53, loss=3.832e-03
  mu=0.01: T_eval=2.590499e-03, e2=5.350e-02, einf=5.711e+00
  mu=0.001: T_eval=2.567154e-03, e2=7.325e-02, einf=6.312e+00
  mu=0.0001: T_eval=2.565582e-03, e2=7.902e-02, einf=6.406e+00
seed=202: T_phi=103.90, T_h=135.22, T_Q=613.97, loss=2.924e-03
  mu=0.01: T_eval=2.747433e-03, e2=3.375e-02, einf=4.846e+00
  mu=0.001: T_eval=2.308482e-03, e2=5.861e-02, einf=6.35